# EDA 02: Source × district coverage audit  *(TODO: Sage)*

**EDA step 2.** For every source we plan to use, which of the 75 districts does it cover,
for which variables and which year? The answer decides *keep / drop / proxy* per variable
(EDA step 6) and feeds the confidence rating beside each ranking.

| | |
|---|---|
| **Inputs** | your downloads under `data/raw/<source_id>/ingest_date=YYYY-MM-DD/`; the spine and name map from notebook 01 |
| **Uses** | `map_to_lgd()` in `src/up_siting/common/names.py`: implement it first (`make todo`) |
| **Outputs** | `data/interim/coverage/coverage_long.parquet`, `.../coverage_grid.parquet`, `docs/eda/02_coverage.md`, new rows in `docs/eda/data-quality-log.md` |
| **Gate** | "source × district coverage grid" (Phase 1) |

Same pattern as notebook 01: each section opens with its question, you fill the empty code
cells (the comments are the pseudocode), and the section's last cell is an `assert` that
must pass. The setup cell below is done for you. Stuck? Ask for a hint before the answer.

In [ ]:
from pathlib import Path

# The "noqa: F401" marks: these imports are used once you fill in the TODO cells; remove the noqa then.
import duckdb  # noqa: F401
import pandas as pd

from up_siting.common.names import map_to_lgd  # noqa: F401
from up_siting.common.paths import INTERIM_DIR, RAW_DIR, SEEDS_DIR  # noqa: F401

spine = pd.read_csv(
    SEEDS_DIR / "lgd_district_spine.csv", dtype={"census_2001_code": str, "census_2011_code": str}
)
name_map = pd.read_csv(SEEDS_DIR / "district_name_map.csv")
lineage = pd.read_csv(SEEDS_DIR / "district_lineage.csv")
OUT_DIR = INTERIM_DIR / "coverage"
print(len(spine), "districts;", len(name_map), "aliases")

## 1. Which sources have we downloaded, and what does each claim to cover?

Register each downloaded file once, here, so every later cell loops over the same list.
One entry per **file** (a source can have several files, e.g. one per year).

In [ ]:
# TODO(sage): one dict per downloaded file. Keys:
#   source_id      snake_case, must match the source_id column in docs/data-sources.md
#   path           Path under RAW_DIR, e.g. RAW_DIR / "census_pca" / "ingest_date=2026-11-25" / "x.csv"
#   reader         "csv" | "xlsx" | "parquet" | "pdf"   (pdf: reuse/extend an extractor first)
#   district_col   the column that holds the district NAME (or code, see section 2)
#   variables      list of columns you care about for the cost model
#   year           reference year of the data (int), not the download date
#   notes          anything odd you noticed while downloading
SOURCES: list[dict] = []

In [ ]:
assert SOURCES, "register at least one source"
required = {"source_id", "path", "reader", "district_col", "variables", "year"}
for s in SOURCES:
    assert required <= s.keys(), (s.get("source_id"), required - s.keys())
    assert Path(s["path"]).is_file(), f"missing file: {s['path']}"
    assert s["variables"], f"{s['source_id']}: list the variables you need"
data_sources_doc = Path("../../docs/data-sources.md").read_text()
undocumented = {s["source_id"] for s in SOURCES if f"`{s['source_id']}`" not in data_sources_doc}
assert not undocumented, f"add these to docs/data-sources.md first: {undocumented}"

## 2. Can every source's districts be mapped to an LGD code?

Load each file, then attach `lgd_district_code` with `map_to_lgd`. Anything unmatched is
either a missing alias (add it to `dbt/seeds/district_aliases.csv` and re-run notebook 01)
or something that isn't a district at all (a "Total" row, a state name), which you
filter out and note.

*If a source already carries Census 2011 codes instead of names, join on
`spine.census_2011_code` instead, and note which districts that loses (see notebook 01, §3).*

In [ ]:
# TODO(sage)
# 1. Write load_source(s) -> pd.DataFrame that reads s["path"] according to s["reader"].
#    csv/parquet: duckdb.sql("select * from read_csv(...)").df() is a good default;
#    read everything as text first (all_varchar=true) and cast later, as in notebook 01.
# 2. For each source: frame = load_source(s); drop non-district rows (totals, state rows);
#    mapped, unmatched = map_to_lgd(frame, s["district_col"], name_map).
# 3. Keep the mapped frames in a dict keyed by source_id (`mapped_by_source`) and the
#    unmatched names in one DataFrame with a source_id column (`unmatched_all`).
mapped_by_source: dict[str, pd.DataFrame] = {}
unmatched_all: pd.DataFrame | None = None

In [ ]:
assert set(mapped_by_source) == {s["source_id"] for s in SOURCES}
assert isinstance(unmatched_all, pd.DataFrame)
# Every unmatched name must be fixed (alias) or consciously excluded (data-quality log).
assert unmatched_all.empty, unmatched_all

## 3. Build the coverage grid: for each source and district, how much data is there?

Two tables. The **long** one is the real result; the **grid** is just a pivot of it for
reading.

* `coverage_long`: one row per (source_id, variable, year, lgd_district_code) with
  `n_rows` and `share_non_null` (0–1). **All 75 districts must appear for every
  source/variable**, including the ones with no data (`share_non_null = 0`), or you can't
  see the gaps. Hint: cross join the spine with the source/variable list, then left join
  the counts.
* `coverage_grid`: index = lgd_district_code (75 rows), one column per `source_id.variable`,
  values = share_non_null.

In [ ]:
# TODO(sage)
# 1. For each mapped source and variable: group by lgd_district_code, count rows and
#    non-null values of the variable.
# 2. Build every (source_id, variable, year) x 75 district combination and left join step 1,
#    filling missing counts with 0. DuckDB SQL over the DataFrames is a good fit here.
# 3. share_non_null = n_non_null / n_rows (0 where n_rows is 0).
# 4. Pivot into coverage_grid.
coverage_long: pd.DataFrame | None = None
coverage_grid: pd.DataFrame | None = None

In [ ]:
assert isinstance(coverage_long, pd.DataFrame) and isinstance(coverage_grid, pd.DataFrame)
expected_cols = {"source_id", "variable", "year", "lgd_district_code", "n_rows", "share_non_null"}
assert expected_cols <= set(coverage_long.columns)
per_combo = coverage_long.groupby(["source_id", "variable", "year"])["lgd_district_code"].nunique()
assert (per_combo == 75).all(), per_combo[per_combo != 75]
assert coverage_long["share_non_null"].between(0, 1).all()
assert len(coverage_grid) == 75 and set(coverage_grid.index) == set(spine["lgd_district_code"])

## 4. Where are the gaps, and are they explained?

Summaries a reviewer can read in ten seconds:

* `by_source`: per source/variable, the number of districts with `share_non_null > 0`.
* `by_district`: per district, how many source/variables have no data, plus `in_census_2011`
  from the spine, because the four post-2011 districts are the expected suspects.

Then look: is a gap explained by a district split (lineage table)? A source that only
covers some regions? A naming problem you missed?

In [ ]:
# TODO(sage)
# 1. by_source: group coverage_long by source_id, variable; count districts with share > 0.
# 2. by_district: count, per district, the source/variables with share == 0; join spine
#    (district_name, in_census_2011); sort by the count, worst first.
# 3. Optional: a heatmap of coverage_grid (districts x sources) with matplotlib's imshow.
by_source: pd.DataFrame | None = None
by_district: pd.DataFrame | None = None

In [ ]:
assert isinstance(by_source, pd.DataFrame) and isinstance(by_district, pd.DataFrame)
assert len(by_district) == 75
assert {"district_name", "in_census_2011"} <= set(by_district.columns)

## 5. Write outputs and record findings

Write `coverage_long` and `coverage_grid` to `OUT_DIR` as Parquet and read them back (the
file is the deliverable). Then fill in the findings below, copy them to
`docs/eda/02_coverage.md`, and add a data-quality log row for every gap or surprise.

In [ ]:
# TODO(sage): mkdir OUT_DIR; write both frames to Parquet; read back with DuckDB and compare row counts.

In [ ]:
assert (OUT_DIR / "coverage_long.parquet").is_file()
assert (OUT_DIR / "coverage_grid.parquet").is_file()

## 6. Findings  *(TODO: Sage)*

1. *Sources registered, and which ones cover all 75 districts …*
2. *Gaps and why (split districts, partial coverage, missing years) …*
3. *Names that needed new aliases …*
4. *Licence issues spotted while downloading …*